# 🧪 FinHelp: THỬ RERANKER LỚN HƠN TRÊN KAGGLE (T4 x2)
Worker gọn chỉ để đo: **không** có Docling, **không** có BGE-M3. Mỗi GPU nạp **hai** reranker:

| Khóa `model` | Mô hình | Vai trò |
|---|---|---|
| `0.6b` | `Qwen/Qwen3-Reranker-0.6B` | đang chạy trong sản phẩm, làm mốc so sánh trong cùng phiên |
| `4b` | `Qwen/Qwen3-Reranker-4B` | ứng viên |

Prompt, cách cắt độ dài (8.192 token) và cách tính điểm P(yes) **giống hệt** `kaggle_gpu_worker.ipynb` và `backend/app/modules/retrieval/reranker.py`. Chỉ khác mô hình.
`/rerank` nhận thêm trường `model` (`"0.6b"` hoặc `"4b"`) và trả thêm `model`, `seconds`. Yêu cầu không có `model` dùng `DEFAULT_MODEL` (mặc định `0.6b`), nên backend trỏ vào worker này vẫn chạy như cũ.
Chọn `DEFAULT_MODEL = "4b"` ở ô 2 nếu muốn thử cảm giác chờ trên giao diện. Lúc đó nhớ đặt `REMOTE_GPU_URL`; worker này không có `/embed`, nên câu hỏi mới cần BGE-M3 sẽ lỗi.

> ⚠️ **CÀI ĐẶT TRÊN KAGGLE (cột bên phải):**
> 1. **ACCELERATOR:** chọn **GPU T4 x2**.
> 2. **INTERNET:** bật **Internet on**.
>
> Tải 4B khoảng 8 GB, lần đầu mất vài phút. Mỗi T4 giữ cả hai mô hình, khoảng 9,5 GB trên 15 GB.
> `Qwen3-Reranker-8B` **không** vừa một T4 ở fp16 (khoảng 16 GB trọng số), nên không thử ở đây.

**Các bước:**
1. Chạy ô 1 → ô 4 theo thứ tự.
2. Ô 3 in bảng kiểm nhanh. Nếu thấy chữ ❌, dừng lại và gửi bảng đó cho tôi.
3. Ô 4 in `URL CỦA BẠN: https://....trycloudflare.com`. Gửi URL đó cho tôi; tôi chạy `experiments/scripts/mvp/rerank_swap.py score --url <URL>` từ máy bạn.
4. Giữ notebook chạy đến khi tôi báo xong (ước tính 1–3 giờ cho 4B; ô 3 cho thời gian thật của một câu hỏi).

In [ ]:
# 1. Cài thư viện và tải cloudflared
!pip install -q fastapi uvicorn pydantic transformers accelerate
!curl -s -Lo cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x cloudflared

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # trước khi import torch: giảm phân mảnh VRAM khi kích thước batch thay đổi
import pathlib
WORK_ROOT = pathlib.Path("/kaggle/working")
WORK_ROOT.mkdir(parents=True, exist_ok=True)
print("✅ Đã cài xong")

In [ ]:
# 2. Nạp hai reranker trên mỗi GPU và dựng API
import math, threading, time
from concurrent.futures import ThreadPoolExecutor
from typing import List, Optional, Tuple

import torch
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from transformers import AutoModelForCausalLM, AutoTokenizer

MODELS = {"0.6b": "Qwen/Qwen3-Reranker-0.6B", "4b": "Qwen/Qwen3-Reranker-4B"}
DEFAULT_MODEL = "0.6b"                       # yêu cầu không ghi "model" (như backend gửi) dùng bản này
BATCH_TOKENS = {"0.6b": 16384, "4b": 8192}    # số token đã đệm tối đa của một lần chạy; 4B nặng hơn nên nửa số đó
SPLIT_MIN_PAIRS = 20                          # ít cặp hơn thì chỉ chạy trên GPU đầu (như worker cũ)

# Giống hệt kaggle_gpu_worker.ipynb và backend/app/modules/retrieval/reranker.py
INSTRUCTION = (
    "Given a question about a company's financial report, judge whether the passage contains information "
    "needed to answer it, such as the relevant financial statement line items and figures or the relevant "
    "narrative disclosure."
)
PREFIX = (
    "<|im_start|>system\nJudge whether the Document meets the requirements based on the Query and the Instruct "
    'provided. Note that the answer can only be "yes" or "no".<|im_end|>\n<|im_start|>user\n'
)
SUFFIX = "<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"
MAX_LEN = 8192

devices = [f"cuda:{i}" for i in range(torch.cuda.device_count())]
assert devices, "Không thấy GPU: chọn ACCELERATOR = GPU T4 x2"
print(f"GPU: {[torch.cuda.get_device_name(i) for i in range(len(devices))]}")

rerankers = {}
for key, name in MODELS.items():
    t0 = time.time()
    tok = AutoTokenizer.from_pretrained(name, padding_side="left")
    models = [AutoModelForCausalLM.from_pretrained(name, dtype=torch.float16, device_map=d).eval() for d in devices]
    rerankers[key] = {
        "name": name, "tok": tok, "models": models,
        "no": tok.convert_tokens_to_ids("no"), "yes": tok.convert_tokens_to_ids("yes"),
        "prefix": tok.encode(PREFIX, add_special_tokens=False), "suffix": tok.encode(SUFFIX, add_special_tokens=False),
    }
    print(f"--> {name}: {len(models)} bản, nạp trong {time.time() - t0:.0f}s")
for i in range(len(devices)):
    free, total = torch.cuda.mem_get_info(i)
    print(f"    GPU {i}: còn {free / 2**30:.1f} / {total / 2**30:.1f} GB")
rerank_lock = threading.Lock()  # mỗi lúc một yêu cầu; mỗi yêu cầu dùng mọi GPU


def _split_by_tokens(lengths, parts):
    # Chia các cặp thành `parts` nhóm có số token gần bằng nhau: dài trước, luôn vào nhóm nhẹ nhất.
    groups, load = [[] for _ in range(parts)], [0] * parts
    for j in sorted(range(len(lengths)), key=lambda j: -lengths[j]):
        k = load.index(min(load))
        groups[k].append(j)
        load[k] += lengths[j]
    return groups


def _score_on(r, model, ids, group, batch_tokens):
    # {chỉ số: P(yes)} của các cặp trong `group`, trên GPU của `model`: xếp theo độ dài, mỗi lần chạy tối đa batch_tokens token đã đệm.
    out = {}

    def flush(batch):
        x = r["tok"].pad({"input_ids": [ids[j] for j in batch]}, padding=True, return_tensors="pt").to(model.device)
        logits = model(**x, logits_to_keep=1).logits[:, -1, :]
        pair = torch.stack([logits[:, r["no"]], logits[:, r["yes"]]], 1).float()
        for j, s in zip(batch, torch.nn.functional.log_softmax(pair, 1)[:, 1].exp().cpu().numpy()):
            out[j] = float(s)

    with torch.inference_mode():
        batch = []
        for j in sorted(group, key=lambda j: len(ids[j])):
            if batch and (len(batch) + 1) * len(ids[j]) > batch_tokens:
                flush(batch)
                batch = []
            batch.append(j)
        if batch:
            flush(batch)
    return out


def score_pairs(key, pairs, instruction=None):
    r = rerankers[key]
    encoded = r["tok"](
        [f"<Instruct>: {instruction or INSTRUCTION}\n<Query>: {q}\n<Document>: {p}" for q, p in pairs],
        padding=False, truncation="longest_first", return_attention_mask=False,
        max_length=MAX_LEN - len(r["prefix"]) - len(r["suffix"]),
    )["input_ids"]
    ids = [r["prefix"] + x + r["suffix"] for x in encoded]
    parts = len(r["models"]) if len(ids) >= SPLIT_MIN_PAIRS else 1
    groups = _split_by_tokens([len(x) for x in ids], parts)
    if parts == 1:
        done = [_score_on(r, r["models"][0], ids, groups[0], BATCH_TOKENS[key])]
    else:
        with ThreadPoolExecutor(max_workers=parts) as pool:
            done = list(pool.map(lambda k: _score_on(r, r["models"][k], ids, groups[k], BATCH_TOKENS[key]), range(parts)))
    scores = {j: s for d in done for j, s in d.items()}
    return [scores[j] for j in range(len(ids))]


app = FastAPI()


class RerankRequest(BaseModel):
    pairs: List[Tuple[str, str]]
    model: Optional[str] = None
    instruction: Optional[str] = None  # thử câu chỉ dẫn khác; bỏ trống = INSTRUCTION của sản phẩm


@app.post("/rerank")
def rerank(req: RerankRequest):
    key = req.model or DEFAULT_MODEL
    if key not in rerankers:
        raise HTTPException(400, f"model phải là một trong {list(rerankers)}")
    if not req.pairs:
        return {"scores": [], "model": key, "seconds": 0.0, "instruction": req.instruction}
    t0 = time.time()
    with rerank_lock:
        scores = score_pairs(key, req.pairs, req.instruction)
    dt = round(time.time() - t0, 3)
    bad = sum(not math.isfinite(s) for s in scores)
    if bad:  # fp16 tràn số: báo lỗi to thay vì trả điểm sai
        raise HTTPException(500, f"{bad} điểm không hữu hạn (fp16 tràn số?) với model {key}")
    print(f"🎯 [{key}] {len(req.pairs)} cặp trong {dt}s")
    return {"scores": scores, "model": key, "seconds": dt, "instruction": req.instruction}


@app.get("/health")
def health():
    gpus = []
    for i in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(i)
        gpus.append({"gpu": i, "name": torch.cuda.get_device_name(i), "vram_free_gib": round(free / 2**30, 2), "vram_total_gib": round(total / 2**30, 2)})
    return {"status": "ready", "models": {k: v["name"] for k, v in rerankers.items()}, "default_model": DEFAULT_MODEL, "gpus": gpus}


print("\n🔥 WORKER ĐÃ KHỞI TẠO XONG!")

In [ ]:
# 3. Kiểm nhanh: điểm hữu hạn, đoạn đúng cao hơn đoạn lạc đề, và thời gian của một câu hỏi
import random

question = "What was the company's total net revenue in fiscal year 2022, and how did it change from 2021?"
good_text = ("CONSOLIDATED STATEMENTS OF OPERATIONS (in millions)\n| | Fiscal 2022 | Fiscal 2021 | Fiscal 2020 |\n|---|---|---|---|\n"
             "| Total net revenue | 394,328 | 365,817 | 274,515 |\n| Cost of sales | 223,546 | 212,981 | 169,559 |\n| Gross margin | 170,782 | 152,836 | 104,956 |\n"
             "Total net revenue increased 8% in fiscal 2022 compared to fiscal 2021, driven by higher sales of products and services.")
other_text = ("The Company's headquarters are located in an office campus it owns. As of the end of the fiscal year the Company had approximately "
              "164,000 full-time equivalent employees. The Company believes its relations with employees are good.")
cases = [(question, good_text), (question, other_text)]

print(f"{'model':>6s} | {'đúng':>6s} {'lạc đề':>7s} | {'1 câu 1 tài liệu (20 cặp)':>26s} | {'1 câu nhiều tài liệu (105 cặp)':>31s} | kết quả")
rng = random.Random(7)
padding = " ".join(["Revenue, operating income, cash flow and segment results are discussed in the following table."] * 40)
for key in rerankers:
    s_good, s_other = score_pairs(key, cases)
    times = {}
    for n in (20, 105):  # cỡ của một câu hỏi thật: đường 1 tài liệu gửi 20 cặp, đường nhiều tài liệu khoảng 7 lượt tìm x 15
        pairs = [(question, good_text + " " + padding[: rng.randint(1500, 3500)]) for _ in range(n)]
        score_pairs(key, pairs[:2])  # làm nóng
        torch.cuda.synchronize()
        t0 = time.time()
        score_pairs(key, pairs)
        torch.cuda.synchronize()
        times[n] = time.time() - t0
    ok = all(math.isfinite(s) for s in (s_good, s_other)) and s_good > s_other
    print(f"{key:>6s} | {s_good:6.3f} {s_other:7.3f} | {times[20]:24.2f} s | {times[105]:29.2f} s | {'✅' if ok else '❌ DỪNG LẠI, gửi bảng này cho tôi'}")
print("\nThời gian ở đây là trên GPU, chưa gồm đường truyền qua tunnel; khoảng 4 nghìn ký tự mỗi đoạn, gần cỡ một đoạn thật (trung vị 741 token).")

In [ ]:
# 4. Khởi động Uvicorn và mở Cloudflare Tunnel (giữ ô này chạy)
import re
import subprocess
import uvicorn


def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")


server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(2)

print("\nĐang khởi tạo Cloudflare Tunnel...")
tunnel_log = WORK_ROOT / "cloudflared.log"  # ghi ra file: pipe đầy mà không đọc thì cloudflared bị treo
tunnel_proc = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
                               stdout=subprocess.DEVNULL, stderr=open(tunnel_log, "w", encoding="utf-8"))

public_url = None
for _ in range(120):
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", tunnel_log.read_text(encoding="utf-8"))
    if match:
        public_url = match.group(0)
        break
    time.sleep(0.5)

if public_url:
    print("\n" + "=" * 70)
    print("⚡ WORKER THỬ RERANKER ĐÃ SẴN SÀNG!")
    print(f"🔗 URL CỦA BẠN: {public_url}")
    print("=" * 70)
    print("\n👉 Gửi URL trên cho Claude. Lệnh đo chạy trên máy bạn:")
    print(f"python experiments/scripts/mvp/rerank_swap.py score --url {public_url}\n")
    try:
        while True:
            time.sleep(60)
            for i in range(torch.cuda.device_count()):
                f, t = torch.cuda.mem_get_info(i)
                print(f"[Heartbeat] GPU {i} VRAM còn {f / 2**30:.2f} / {t / 2**30:.2f} GB")
    except KeyboardInterrupt:
        print("Dừng server.")
else:
    print("❌ Không lấy được URL tunnel; xem", tunnel_log)

In [ ]:
# 5. CHỈ CHẠY nếu phiên này mở từ bản notebook TRƯỚC khi có trường "instruction" (chạy lại cũng vô hại)
# Cách làm: bấm ⏹ (Interrupt) ô 4. Server và tunnel vẫn chạy, URL giữ nguyên. Rồi chạy ô này: nó thay /rerank
# bằng bản nhận thêm "instruction" và tiếp tục nhịp heartbeat như ô 4.
def score_pairs(key, pairs, instruction=None):
    r = rerankers[key]
    encoded = r["tok"](
        [f"<Instruct>: {instruction or INSTRUCTION}\n<Query>: {q}\n<Document>: {p}" for q, p in pairs],
        padding=False, truncation="longest_first", return_attention_mask=False,
        max_length=MAX_LEN - len(r["prefix"]) - len(r["suffix"]),
    )["input_ids"]
    ids = [r["prefix"] + x + r["suffix"] for x in encoded]
    parts = len(r["models"]) if len(ids) >= SPLIT_MIN_PAIRS else 1
    groups = _split_by_tokens([len(x) for x in ids], parts)
    if parts == 1:
        done = [_score_on(r, r["models"][0], ids, groups[0], BATCH_TOKENS[key])]
    else:
        with ThreadPoolExecutor(max_workers=parts) as pool:
            done = list(pool.map(lambda k: _score_on(r, r["models"][k], ids, groups[k], BATCH_TOKENS[key]), range(parts)))
    scores = {j: s for d in done for j, s in d.items()}
    return [scores[j] for j in range(len(ids))]


class RerankRequest(BaseModel):
    pairs: List[Tuple[str, str]]
    model: Optional[str] = None
    instruction: Optional[str] = None


def rerank(req: RerankRequest):
    key = req.model or DEFAULT_MODEL
    if key not in rerankers:
        raise HTTPException(400, f"model phải là một trong {list(rerankers)}")
    if not req.pairs:
        return {"scores": [], "model": key, "seconds": 0.0, "instruction": req.instruction}
    t0 = time.time()
    with rerank_lock:
        scores = score_pairs(key, req.pairs, req.instruction)
    dt = round(time.time() - t0, 3)
    bad = sum(not math.isfinite(s) for s in scores)
    if bad:
        raise HTTPException(500, f"{bad} điểm không hữu hạn (fp16 tràn số?) với model {key}")
    print(f"🎯 [{key}{' +chỉ dẫn' if req.instruction else ''}] {len(req.pairs)} cặp trong {dt}s")
    return {"scores": scores, "model": key, "seconds": dt, "instruction": req.instruction}


app.router.routes = [r for r in app.router.routes if getattr(r, "path", None) != "/rerank"]
app.post("/rerank")(rerank)
print("✅ /rerank đã nhận trường instruction. URL không đổi:", public_url)
try:
    while True:
        time.sleep(60)
        for i in range(torch.cuda.device_count()):
            f, t = torch.cuda.mem_get_info(i)
            print(f"[Heartbeat] GPU {i} VRAM còn {f / 2**30:.2f} / {t / 2**30:.2f} GB")
except KeyboardInterrupt:
    print("Dừng heartbeat (server vẫn chạy).")
